In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from catboost import CatBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score
import time
import shap
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.decomposition import PCA

c:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class OutlierHandler(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.5):
        self.factor = factor
        self.lower_bounds = {}
        self.upper_bounds = {}

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        for col in X_df.columns:
            Q1 = X_df[col].quantile(0.25)
            Q3 = X_df[col].quantile(0.75)
            IQR = Q3 - Q1
            self.lower_bounds[col] = Q1 - self.factor * IQR
            self.upper_bounds[col] = Q3 + self.factor * IQR
        return self

    def transform(self, X, y=None):
        X_df = pd.DataFrame(X).copy()
        for col in X_df.columns:
            # Capping (Winsorization)
            X_df[col] = np.where(X_df[col] < self.lower_bounds[col], self.lower_bounds[col], X_df[col])
            X_df[col] = np.where(X_df[col] > self.upper_bounds[col], self.upper_bounds[col], X_df[col])
        return X_df.values

In [3]:
df = pd.read_csv('./dataset/Oktay_kg/heart_preprocess.csv')

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Unnamed: 0            10000 non-null  int64  
 1   Age                   9971 non-null   float64
 2   Gender                9981 non-null   float64
 3   Blood Pressure        9981 non-null   float64
 4   Cholesterol Level     9970 non-null   float64
 5   Exercise Habits       9975 non-null   float64
 6   Smoking               9975 non-null   float64
 7   Family Heart Disease  9979 non-null   float64
 8   Diabetes              9970 non-null   float64
 9   BMI                   9978 non-null   float64
 10  High Blood Pressure   9974 non-null   float64
 11  Low HDL Cholesterol   9975 non-null   float64
 12  High LDL Cholesterol  9974 non-null   float64
 13  Alcohol Consumption   7414 non-null   float64
 14  Stress Level          9978 non-null   float64
 15  Sleep Hours         

In [6]:
df['Heart Disease Status'].value_counts()

Heart Disease Status
0    8000
1    2000
Name: count, dtype: int64

In [7]:
X = df.drop('Heart Disease Status', axis=1)
y = df['Heart Disease Status']

X_train,X_test,Y_train,Y_test = train_test_split(X,y,test_size=0.2,stratify=y)


num_features = [
    "Age",
    "Blood Pressure",
    "Cholesterol Level",
    "BMI",
    "Sleep Hours",
    "Triglyceride Level",
    "Fasting Blood Sugar",
    "CRP Level",
    "Homocysteine Level"
]
cat_features = [
    "Gender",             
    "Smoking",
    "Family Heart Disease",
    "Diabetes",
    "High Blood Pressure",
    "Low HDL Cholesterol",
    "High LDL Cholesterol",
    "Exercise Habits",       
    "Alcohol Consumption",   
    "Stress Level",          
    "Sugar Consumption"      
    ]

In [11]:
Y_train.value_counts()

Heart Disease Status
0    6400
1    1600
Name: count, dtype: int64

PREPROCESSING DATA KNN-SVM-LR

In [ ]:
num_pipeline_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("outlier", OutlierHandler()),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocess_scaled = ColumnTransformer([
    ("num", num_pipeline_scaled, num_features),
    ("cat", cat_pipeline, cat_features)
])

Decision Tree, Random Forest, Bagging

In [ ]:
num_pipeline_no_scale = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

preprocess_tree = ColumnTransformer([
    ("num", num_pipeline_no_scale, num_features),
    ("cat", cat_pipeline, cat_features)
])

XGBoost, AdaBoost

In [ ]:
preprocess_boost = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), num_features),
    
    ("cat", cat_pipeline, cat_features)
])

In [ ]:
CatBoostClassifier(cat_features=cat_features)

CatBoostClassifier(cat_features=['Gender', 'Smoking', 'Family Heart Disease', 'Diabetes', 'High Blood Pressure', 'Low HDL Cholesterol', 'High LDL Cholesterol', 'Exercise Habits', 'Alcohol Consumption', 'Stress Level', 'Sugar Consumption'])

In [ ]:
pipe_knn = Pipeline([
    ("preprocess", preprocess_scaled),
    ("model", KNeighborsClassifier())
])

pipe_lr = Pipeline([
    ("preprocess", preprocess_scaled),
    ("model", LogisticRegression(max_iter=1000))
])

pipe_svm = Pipeline([
    ("preprocess", preprocess_scaled),
    ("model", SVC(probability=True))
])

pipe_dt = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", DecisionTreeClassifier())
])

pipe_rf = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", RandomForestClassifier())
])

pipe_bagging = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=100
    ))
])

pipe_xgb = Pipeline([
    ("preprocess", preprocess_boost),
    ("model", XGBClassifier(eval_metric='logloss'))
])

pipe_ada = Pipeline([
    ("preprocess", preprocess_boost),
    ("model", AdaBoostClassifier())
])

pipe_cat = Pipeline([
    ("preprocess", preprocess_boost),
    ("model", CatBoostClassifier(verbose=0))
])

In [ ]:
pipelines = {
    "KNN": pipe_knn,
    "LR": pipe_lr,
    "SVM": pipe_svm,
    "DT": pipe_dt,
    "RF": pipe_rf,
    "Bagging": pipe_bagging,
    "XGB": pipe_xgb,
    "Ada": pipe_ada,
    "CatBoost": pipe_cat
}

In [ ]:
param_knn = {
    "model__n_neighbors": [3,5,7,9,11],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

param_lr = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs"]
}

param_svm = {
    "model__C": [0.1, 1, 10, 100],
    "model__kernel": ["rbf", "linear"],
    "model__gamma": ["scale", "auto"]
}

param_dt = {
    "model__max_depth": [3,5,10,20,None],
    "model__min_samples_split": [2,5,10],
    "model__min_samples_leaf": [1,2,4],
    "model__criterion": ["gini", "entropy"]
}

param_rf = {
    "model__n_estimators": [100,200,300],
    "model__max_depth": [None,5,10,20],
    "model__min_samples_split": [2,5],
    "model__min_samples_leaf": [1,2],
    "model__max_features": ["sqrt", "log2"]
}

param_bagging = {
    "model__n_estimators": [50,100,200],
    "model__max_samples": [0.5, 0.7, 1.0],
    "model__max_features": [0.5, 0.7, 1.0]
}

param_xgb = {
    "model__n_estimators": [100,200,300],
    "model__max_depth": [3,5,7],
    "model__learning_rate": [0.01, 0.1, 0.2],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

param_ada = {
    "model__n_estimators": [50,100,200],
    "model__learning_rate": [0.01, 0.1, 1]
}

param_cat = {
    "model__iterations": [100,200,300],
    "model__depth": [4,6,8],
    "model__learning_rate": [0.01, 0.1],
    "model__l2_leaf_reg": [1,3,5]
}

In [ ]:
param_grids = {
    "KNN": param_knn,
    "LR": param_lr,
    "SVM": param_svm,
    "DT": param_dt,
    "RF": param_rf,
    "Bagging": param_bagging,
    "XGB": param_xgb,
    "Ada": param_ada,
    "CatBoost": param_cat
}

In [ ]:
best_models = {}
evaluation_results = []

for name, pipe in pipelines.items():
    if name == 'SVM':
        pass
    print(f"[{name}] Đang huấn luyện với GridSearchCV...")
    
    # 1. Huấn luyện GridSearchCV
    grid = GridSearchCV(
        pipe,
        param_grids[name],
        cv=5,
        scoring="neg_log_loss",
        n_jobs=-1,
        return_train_score=True,
        verbose=1
    )
    
    start_time = time.time()
    
    grid.fit(X_train, Y_train)
    
    end_time = time.time()
    training_time = end_time - start_time
    
    best_models[name] = grid.best_estimator_
    
    
    best_idx = grid.best_index_
    
    # neg_log_loss trả về số âm, nên ta nhân -1 để có giá trị log_loss chuẩn (càng nhỏ càng tốt)
    train_loss = -grid.cv_results_['mean_train_score'][best_idx]
    val_loss_cv = -grid.cv_results_['mean_test_score'][best_idx]
    
    # 2. Dự đoán xác suất cho nhãn Positive (1)
    # SVM của bạn đã có probability=True nên predict_proba sẽ chạy bình thường
    y_probs = grid.best_estimator_.predict_proba(X_test)[:, 1]
    
    # 3. Tính toán ROC AUC và tìm ngưỡng cắt (Threshold) tối ưu
    auc = roc_auc_score(Y_test, y_probs)
    fpr, tpr, thresholds = roc_curve(Y_test, y_probs)
    
    # Sử dụng Youden's J statistic để tìm threshold cắt tối ưu (tối đa hoá tpr - fpr)
    J = tpr - fpr
    optimal_idx = np.argmax(J)
    optimal_threshold = thresholds[optimal_idx]
    
    # 4. Dự đoán dựa trên ngưỡng vừa tìm được
    y_pred = (y_probs >= optimal_threshold).astype(int)
    
    # 5. Đánh giá các độ đo khác
    acc = accuracy_score(Y_test, y_pred)
    prec = precision_score(Y_test, y_pred, zero_division=0)
    rec = recall_score(Y_test, y_pred, zero_division=0)
    
    evaluation_results.append({
        "Model": name,
        "Training Time (s)": round(training_time, 4),
        "AUC": auc,
        "Optimal Threshold": optimal_threshold,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Best Params": grid.best_params_
    })

# Tạo DataFrame hiển thị kết quả
results_df = pd.DataFrame(evaluation_results)
results_df.sort_values(by="AUC", ascending=False, inplace=True)
results_df.reset_index(drop=True, inplace=True)

display(results_df)

In [ ]:
smote_step = SMOTE(random_state=42)

pipe_knn = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("model", KNeighborsClassifier())
])

pipe_lr = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("model", LogisticRegression(max_iter=1000))
])

pipe_svm = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("model", SVC(probability=True))
])

pipe_dt = ImbPipeline([
    ("preprocess", preprocess_tree),
    ("smote", smote_step),
    ("model", DecisionTreeClassifier(random_state=42))
])

pipe_rf = ImbPipeline([
    ("preprocess", preprocess_tree),
    ("smote", smote_step),
    ("model", RandomForestClassifier(random_state=42))
])

pipe_bagging = ImbPipeline([
    ("preprocess", preprocess_tree),
    ("smote", smote_step),
    ("model", BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=100,
        random_state=42
    ))
])

pipe_xgb = ImbPipeline([
    ("preprocess", preprocess_boost),
    ("smote", smote_step),
    ("model", XGBClassifier(eval_metric='logloss', random_state=42))
])

pipe_ada = ImbPipeline([
    ("preprocess", preprocess_boost),
    ("smote", smote_step),
    ("model", AdaBoostClassifier(random_state=42))
])

pipe_cat = ImbPipeline([
    ("preprocess", preprocess_boost),
    ("smote", smote_step),
    ("model", CatBoostClassifier(verbose=0, random_state=42))
])

In [ ]:
best_models = {}
evaluation_results = []

for name, pipe in pipelines.items():
    if name == 'SVM':
        pass
    print(f"[{name}] Đang huấn luyện với GridSearchCV...")
    
    # 1. Huấn luyện GridSearchCV
    grid = GridSearchCV(
        pipe,
        param_grids[name],
        cv=5,
        scoring="neg_log_loss",
        n_jobs=-1,
        return_train_score=True,
        verbose=1
    )
    
    start_time = time.time()
    
    grid.fit(X_train, Y_train)
    
    end_time = time.time()
    training_time = end_time - start_time
    
    best_models[name] = grid.best_estimator_
    
    
    best_idx = grid.best_index_
    
    # neg_log_loss trả về số âm, nên ta nhân -1 để có giá trị log_loss chuẩn (càng nhỏ càng tốt)
    train_loss = -grid.cv_results_['mean_train_score'][best_idx]
    val_loss_cv = -grid.cv_results_['mean_test_score'][best_idx]
    
    # 2. Dự đoán xác suất cho nhãn Positive (1)
    # SVM của bạn đã có probability=True nên predict_proba sẽ chạy bình thường
    y_probs = grid.best_estimator_.predict_proba(X_test)[:, 1]
    
    # 3. Tính toán ROC AUC và tìm ngưỡng cắt (Threshold) tối ưu
    auc = roc_auc_score(Y_test, y_probs)
    fpr, tpr, thresholds = roc_curve(Y_test, y_probs)
    
    # Sử dụng Youden's J statistic để tìm threshold cắt tối ưu (tối đa hoá tpr - fpr)
    J = tpr - fpr
    optimal_idx = np.argmax(J)
    optimal_threshold = thresholds[optimal_idx]
    
    # 4. Dự đoán dựa trên ngưỡng vừa tìm được
    y_pred = (y_probs >= optimal_threshold).astype(int)
    
    # 5. Đánh giá các độ đo khác
    acc = accuracy_score(Y_test, y_pred)
    prec = precision_score(Y_test, y_pred, zero_division=0)
    rec = recall_score(Y_test, y_pred, zero_division=0)
    
    evaluation_results.append({
        "Model": name,
        "Training Time (s)": round(training_time, 4),
        "AUC": auc,
        "Optimal Threshold": optimal_threshold,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Best Params": grid.best_params_
    })

# Tạo DataFrame hiển thị kết quả
results_df = pd.DataFrame(evaluation_results)
results_df.sort_values(by="AUC", ascending=False, inplace=True)
results_df.reset_index(drop=True, inplace=True)

display(results_df)

In [ ]:

pca_step = PCA(random_state=42) # Để trống tham số này để Param Grid tự tune

# TẤT CẢ các model đều phải dùng preprocess_scaled vì có PCA
pipe_knn = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("pca", pca_step),
    ("model", KNeighborsClassifier())
])

pipe_lr = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("pca", pca_step),
    ("model", LogisticRegression(max_iter=1000))
])

pipe_svm = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("pca", pca_step),
    ("model", SVC(probability=True))
])

pipe_dt = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("pca", pca_step),
    ("model", DecisionTreeClassifier(random_state=42))
])

pipe_rf = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("pca", pca_step),
    ("model", RandomForestClassifier(random_state=42))
])

pipe_bagging = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("pca", pca_step),
    ("model", BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=100,
        random_state=42
    ))
])

pipe_xgb = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("pca", pca_step),
    ("model", XGBClassifier(eval_metric='logloss', random_state=42))
])

pipe_ada = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("pca", pca_step),
    ("model", AdaBoostClassifier(random_state=42))
])

pipe_cat = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("pca", pca_step),
    ("model", CatBoostClassifier(verbose=0, random_state=42))
])

pipelines = {
    "KNN": pipe_knn,
    "LR": pipe_lr,
    "SVM": pipe_svm,
    "DT": pipe_dt,
    "RF": pipe_rf,
    "Bagging": pipe_bagging,
    "XGB": pipe_xgb,
    "Ada": pipe_ada,
    "CatBoost": pipe_cat
}

In [ ]:
param_knn = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__n_neighbors": [3, 5, 7, 9, 11],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

param_lr = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs"]
}

param_svm = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__C": [0.1, 1, 10, 100],
    "model__kernel": ["rbf", "linear"],
    "model__gamma": ["scale", "auto"]
}

param_dt = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__max_depth": [3, 5, 10, 20, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__criterion": ["gini", "entropy"]
}

param_rf = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 5, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

param_bagging = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__n_estimators": [50, 100, 200],
    "model__max_samples": [0.5, 0.7, 1.0],
    "model__max_features": [0.5, 0.7, 1.0]
}

param_xgb = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.01, 0.1, 0.2],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

param_ada = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__n_estimators": [50, 100, 200],
    "model__learning_rate": [0.01, 0.1, 1]
}

param_cat = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__iterations": [100, 200, 300],
    "model__depth": [4, 6, 8],
    "model__learning_rate": [0.01, 0.1]
    # Bỏ l2_leaf_reg đi bớt để giảm thời gian train nếu bạn kết hợp cả PCA
}

param_grids = {
    "KNN": param_knn,
    "LR": param_lr,
    "SVM": param_svm,
    "DT": param_dt,
    "RF": param_rf,
    "Bagging": param_bagging,
    "XGB": param_xgb,
    "Ada": param_ada,
    "CatBoost": param_cat
}

In [ ]:
best_models = {}
evaluation_results = []

for name, pipe in pipelines.items():
    if name == 'SVM':
        pass
    print(f"[{name}] Đang huấn luyện với GridSearchCV...")
    
    # 1. Huấn luyện GridSearchCV
    grid = GridSearchCV(
        pipe,
        param_grids[name],
        cv=5,
        scoring="neg_log_loss",
        n_jobs=-1,
        return_train_score=True,
        verbose=1
    )
    
    start_time = time.time()
    
    grid.fit(X_train, Y_train)
    
    end_time = time.time()
    training_time = end_time - start_time
    
    best_models[name] = grid.best_estimator_
    
    
    best_idx = grid.best_index_
    
    # neg_log_loss trả về số âm, nên ta nhân -1 để có giá trị log_loss chuẩn (càng nhỏ càng tốt)
    train_loss = -grid.cv_results_['mean_train_score'][best_idx]
    val_loss_cv = -grid.cv_results_['mean_test_score'][best_idx]
    
    # 2. Dự đoán xác suất cho nhãn Positive (1)
    # SVM của bạn đã có probability=True nên predict_proba sẽ chạy bình thường
    y_probs = grid.best_estimator_.predict_proba(X_test)[:, 1]
    
    # 3. Tính toán ROC AUC và tìm ngưỡng cắt (Threshold) tối ưu
    auc = roc_auc_score(Y_test, y_probs)
    fpr, tpr, thresholds = roc_curve(Y_test, y_probs)
    
    # Sử dụng Youden's J statistic để tìm threshold cắt tối ưu (tối đa hoá tpr - fpr)
    J = tpr - fpr
    optimal_idx = np.argmax(J)
    optimal_threshold = thresholds[optimal_idx]
    
    # 4. Dự đoán dựa trên ngưỡng vừa tìm được
    y_pred = (y_probs >= optimal_threshold).astype(int)
    
    # 5. Đánh giá các độ đo khác
    acc = accuracy_score(Y_test, y_pred)
    prec = precision_score(Y_test, y_pred, zero_division=0)
    rec = recall_score(Y_test, y_pred, zero_division=0)
    
    evaluation_results.append({
        "Model": name,
        "Training Time (s)": round(training_time, 4),
        "AUC": auc,
        "Optimal Threshold": optimal_threshold,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Best Params": grid.best_params_
    })

# Tạo DataFrame hiển thị kết quả
results_df = pd.DataFrame(evaluation_results)
results_df.sort_values(by="AUC", ascending=False, inplace=True)
results_df.reset_index(drop=True, inplace=True)

display(results_df)